In [1]:
from csrio_image2biomass.configs.settings import AUGUMENTED_DATA_DIR
import polars as pl
test = pl.read_csv(AUGUMENTED_DATA_DIR / "test.csv")
test.sort("image_path").head()

image_path,Dry_Clover_g,Dry_Dead_g,Dry_Green_g,Dry_Total_g,GDM_g
str,i64,i64,i64,i64,i64
"""test/ID1001187975.jpg""",0,0,0,0,0


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
from PIL import Image
import numpy as np
import os
from typing import Dict, Any, Tuple

class BiomassDataset(Dataset):
    def __init__(self, dataframe: pl.DataFrame, img_dir: str):
        self.dataframe = dataframe
        self.img_dir = img_dir
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
        ])

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx) -> Tuple[torch.Tensor, torch.Tensor]:
        item: Dict[str, Any] = self.dataframe.row(idx, named=True)
        img_name = os.path.join(self.img_dir, item['image_path'])
        image = Image.open(img_name).convert('RGB')
        labels = np.array([item['Dry_Clover_g'], item['Dry_Dead_g'], item['Dry_Green_g'], item['Dry_Total_g'], item['GDM_g']], dtype=np.float32)
        image = self.transform(image)
        labels = torch.tensor(labels, dtype=torch.float32)

        return image, labels
    
test_dataset = BiomassDataset(dataframe=test, img_dir=str(AUGUMENTED_DATA_DIR))
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=12)

for batch in test_loader:
    images, labels = batch
    print(f"Image batch shape: {images.size()}")
    print(f"Label batch shape: {labels.size()}")
    break


Image batch shape: torch.Size([1, 3, 224, 224])
Label batch shape: torch.Size([1, 5])


In [5]:
# Load the model architecture (ResNet18 with custom output layer)
model = models.resnet50(pretrained=False)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 5)  # 5 outputs for the biomass predictions

# Load the state dictionary
state_dict = torch.load("biomass_model.pth", map_location=torch.device('cpu'))
model.load_state_dict(state_dict)
model.eval()  # Set to evaluation mode

outputs = []
with torch.no_grad():
    for images, labels in test_loader:
        preds = model(images)
        preds = preds.cpu()
        outputs.append(preds)

outputs = torch.cat(outputs, dim=0).numpy()
outputs

/home/administrator/Desktop/datascience/kaggle/csrio-image2biomass/.venv/lib/python3.13/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/administrator/Desktop/datascience/kaggle/csrio-image2biomass/.venv/lib/python3.13/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


array([[ 1.5383393,  5.249547 ,  5.4993615, 11.997256 ,  6.7781267]],
      dtype=float32)

In [6]:
output_df = pl.DataFrame(outputs, schema=["Dry_Clover_g", "Dry_Dead_g", "Dry_Green_g", "Dry_Total_g", "GDM_g"])
output_df

Dry_Clover_g,Dry_Dead_g,Dry_Green_g,Dry_Total_g,GDM_g
f32,f32,f32,f32,f32
1.538339,5.249547,5.499362,11.997256,6.778127


In [7]:
test_df = test.select("image_path").hstack(output_df).unpivot(on=["Dry_Clover_g", "Dry_Dead_g", "Dry_Green_g", "Dry_Total_g", "GDM_g"], index="image_path", value_name="target")
test_df

image_path,variable,target
str,str,f32
"""test/ID1001187975.jpg""","""Dry_Clover_g""",1.538339
"""test/ID1001187975.jpg""","""Dry_Dead_g""",5.249547
"""test/ID1001187975.jpg""","""Dry_Green_g""",5.499362
"""test/ID1001187975.jpg""","""Dry_Total_g""",11.997256
"""test/ID1001187975.jpg""","""GDM_g""",6.778127


In [10]:
# concat image_path and variable columns
submission = test_df.with_columns(
    pl.concat_str([pl.col("image_path").str.split('/').list.get(-1).str.split('.jpg').list.get(0), pl.lit("__"), pl.col("variable")]).alias("sample_id")
).select(["sample_id", "target"])
print(submission)

shape: (5, 2)
┌────────────────────────────┬───────────┐
│ sample_id                  ┆ target    │
│ ---                        ┆ ---       │
│ str                        ┆ f32       │
╞════════════════════════════╪═══════════╡
│ ID1001187975__Dry_Clover_g ┆ 1.538339  │
│ ID1001187975__Dry_Dead_g   ┆ 5.249547  │
│ ID1001187975__Dry_Green_g  ┆ 5.499362  │
│ ID1001187975__Dry_Total_g  ┆ 11.997256 │
│ ID1001187975__GDM_g        ┆ 6.778127  │
└────────────────────────────┴───────────┘


In [9]:
submission.write_csv("submission.csv")